# EDA para la defensa — cupones personalizados

Este notebook **no calcula nada**: presenta las tablas y gráficos generados por
`python -m src.analizar_repeticion` (o la etapa `eda_defensa` del pipeline).
La fuente reproducible es el script; acá se muestra la lectura de cada resultado.

Datos: entrenamiento + validación. El conjunto de prueba no se usa.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

# El notebook vive en notebooks/; los resultados en outputs/eda/.
EDA = Path.cwd().parent / "outputs" / "eda" if Path.cwd().name == "notebooks" else Path("outputs") / "eda"
resumen = json.loads((EDA / "defensa_resumen.json").read_text(encoding="utf-8"))
resumen

## 1. ¿Cuánto de cada canasta es recompra?

Sin contar la primera canasta, **la mitad de los productos de una canasta ya
habían sido comprados antes por el hogar** (≈ 50 %). La proporción crece con la
antigüedad del cliente: ≈ 12 % en las canastas 2 a 5 y ≈ 64 % a partir de la
canasta 101.

Si se considera "ya comprado" a cualquier variante del mismo producto (misma
subcategoría, marca y tamaño), la recompra sube a ≈ 61 %: parte de lo que por
código parece nuevo es el mismo producto en otra presentación. Esto respalda la
regla que descarta variantes en explore (D-008).

**Lectura para la defensa:** la recompra es la mitad de lo que compra un hogar,
por eso repeat es un objetivo propio y fácil de acertar; la otra mitad es
novedad, por eso existe explore. Son problemas de distinta dificultad y se
evalúan por separado.

In [ ]:
display(pd.read_csv(EDA / "defensa_repeticion_por_tramo.csv"))
display(Image(EDA / "defensa_repeticion_por_canasta.png"))

## 2. Objetivos de validación por hogar

De 2.491 hogares, 2.033 recompraron algo en su canasta de validación y 2.085
compraron algo nuevo; 1.627 hicieron ambas cosas. Todos los hogares tienen al
menos un objetivo. Por eso las métricas repeat y explore se promedian sobre
cantidades distintas de hogares.

In [ ]:
pd.DataFrame([resumen["objetivos_validacion"]])

## 3. Rendimiento según la longitud del historial (cold start)

Resultado contraintuitivo y útil:

- **Repeat rinde parecido en todos los segmentos** (Hit Rate@10 ≈ 63–67 %):
  frecuencia + recencia funciona aun con pocas canastas.
- **Explore rinde mucho mejor en hogares con historial corto**: ALS acierta en
  ≈ 16 % de los hogares con 1–48 canastas y en ≈ 3 % de los que tienen más de
  116. Un cliente nuevo todavía no compró los productos comunes, así que hay
  más novedades "fáciles"; un cliente antiguo ya probó casi todo lo habitual.
- ALS supera a popularidad en los tres segmentos.

**Mejora futura (no aplicada):** la proporción repeat/explore podría depender
de la antigüedad del hogar (más explore para clientes nuevos). No se ajustó
porque la configuración ya está congelada y la prueba fue usada (D-006, D-007).

Nota: el modelo excluye hogares con menos de 3 canastas; el cold start total
(hogar sin historial) se cubre en el prototipo con una lista de respaldo.

In [ ]:
display(pd.read_csv(EDA / "defensa_rendimiento_por_historial.csv")[["lista", "segmento", "hogares_evaluables", "hit_rate", "ndcg"]])
display(Image(EDA / "defensa_rendimiento_por_historial.png"))

## 4. Concentración de la popularidad y cobertura del catálogo

- El **1 % de los productos** reúne ≈ 35 % de las apariciones en canastas; el
  **10 %**, ≈ 77 %. Gini ≈ 0,84. Más de 33.000 productos aparecen en una sola
  canasta.
- Por eso popularidad recomienda casi siempre lo mismo (81 productos distintos)
  y ALS cubre mucho más del catálogo (≈ 1.050), en todos los departamentos.

In [ ]:
display(Image(EDA / "defensa_concentracion_popularidad.png"))
display(pd.read_csv(EDA / "defensa_cobertura_por_departamento.csv").head(12))
display(Image(EDA / "defensa_cobertura_por_departamento.png"))

## 5. Combustible y tabaco

- **Combustible:** lo compra el 54 % de los hogares y el 83 % de ellos lo
  recompra. Sin reglas aparecía en la lista de **45 % de los hogares**: un
  acierto fácil, pero no ofrecible con cupón.
- **Tabaco:** lo compra el 45 % de los hogares y el 71 % lo recompra; sin reglas
  aparecía en la lista del 8 % de los hogares. Su promoción está prohibida
  (en Uruguay, Ley 18.256).

Estos números justifican las exclusiones de D-008 y explican por qué el Hit Rate
baja al aplicar reglas sin que el modelo empeore.

In [ ]:
pd.read_csv(EDA / "defensa_combustible_y_tabaco.csv")